In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import joblib

In [3]:
file_add = "/content/drive/MyDrive/apadamitra_heatwave_dataset.csv"
df = pd.read_csv(file_add)

In [4]:
df.head()

,wind_speed,cloud_cover,pressure_surface_level,dew_point,uv_index,max_temperature,min_temperature,max_humidity,min_humidity,heatwave
0,2.785259,21.462155,993.455514,29.974869,7.757285,33.755248,32.350385,69.260757,61.289844,0.0
1,9.399230,49.889837,1020.945313,25.815398,3.376130,32.172688,27.963711,81.054374,75.400783,0.0
2,11.043974,94.023956,1036.157480,35.277626,6.317181,40.162249,35.729240,94.553372,87.829747,0.0
3,14.575710,8.239687,1034.623731,33.325332,11.000000,39.520985,33.959291,82.118124,73.548256,0.0
4,6.658576,49.889837,966.704193,22.955294,3.825826,27.914785,19.918764,51.278517,37.825318,0.0


In [6]:
df.shape

(57428, 10)

In [7]:
X = df.drop("heatwave", axis=1)
y = df["heatwave"]

In [8]:
print("Class distribution:")
print(y.value_counts())

Class distribution:
heatwave
0.0    52533
1.0     4895
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [12]:
print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())


After SMOTE:
heatwave
0.0    42026
1.0    42026
Name: count, dtype: int64


In [13]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

In [14]:
model.fit(
    X_train_smote,
    y_train_smote
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
pred = model.predict(X_test_scaled)

In [17]:
print("\nAccuracy:")
print(accuracy_score(y_test,pred))


Accuracy:
0.9661326832665854


In [18]:
print("\nClassification Report:")
print(classification_report(y_test,pred))


Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98     10507
         1.0       0.74      0.94      0.83       979

    accuracy                           0.97     11486
   macro avg       0.87      0.95      0.90     11486
weighted avg       0.97      0.97      0.97     11486



In [19]:
joblib.dump(
    model,
    "apadamitra_heatwave_model.pkl"
)
joblib.dump(
    scaler,
    "apadamitra_headwave_scaler.pkl"
)

['apadamitra_headwave_scaler.pkl']